In [ ]:
"""
fig3_coupling_decomposition.py
-------------------------------
Generates Fig. 3 of the paper (Künneth coupling decomposition validation)
from synthetic data, following Algorithm 1 (Synthetic Community Generator)
and the statistical framework of Theorem 2.

Physical interpretation
-----------------------
In the paper, the coupling residual test statistic is

    R_hat = ||h_tensor_hat||^2 - ||h1_hat||^2 · ||h2_hat||^2

and the paper proves  E[R_hat] = R_target + d_r · σ²,
where R_target = ||r*||² is the injected residual norm and d_r is the
dimension of the coupling-residual subspace.

For SCALAR (m=1) sheaves on the same complex, the simplest faithful
validation is the harmonic ENERGY RECOVERY test:

    R_hat = ||h_hat||²   (estimated ||·||² energy of the harmonic component)
    R_target = ||h*||²   (true injected harmonic norm)
    E[R_hat] = R_target + b1 · σ²     (non-central chi² expectation)

where b1 = dim ker Δ1 is the first Betti number of K.
This is exactly the statistical content the figure illustrates.

Dependencies: numpy, scipy, networkx, matplotlib
"""

import numpy as np
import scipy.linalg as la
import scipy.sparse as sp
import networkx as nx
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import stats
from itertools import combinations

# ─────────────────────────────────────────────────────────────────────────────
# 1.  SIMPLICIAL COMPLEX  (guaranteed b1 ≥ 1)
# ─────────────────────────────────────────────────────────────────────────────

def build_complex(n, p_edge, n_triangles, rng):
    """
    Connected Erdős–Rényi graph; add at most n_triangles graph-triangles
    so that b1 = ne - nv + 1 - nt ≥ 1.
    """
    for _ in range(100_000):
        G = nx.erdos_renyi_graph(n, p_edge,
                                 seed=int(rng.integers(0, 2**31)))
        if nx.is_connected(G) and G.number_of_edges() >= n:
            break

    vertices = sorted(G.nodes())
    edges    = sorted(tuple(sorted(e)) for e in G.edges())
    edge_set = set(edges)
    nv, ne   = len(vertices), len(edges)

    all_tri = [
        (i, j, k)
        for i, j, k in combinations(vertices, 3)
        if (i,j) in edge_set and (i,k) in edge_set and (j,k) in edge_set
    ]

    max_t  = max(0, ne - nv)          # ensures b1 = ne-nv+1-nt ≥ 1
    n_add  = min(n_triangles, max_t, len(all_tri))
    if n_add > 0:
        idx       = rng.choice(len(all_tri), size=n_add, replace=False)
        triangles = [all_tri[i] for i in sorted(idx)]
    else:
        triangles = []

    b1 = ne - nv + 1 - len(triangles)
    return vertices, edges, triangles, b1


# ─────────────────────────────────────────────────────────────────────────────
# 2.  COBOUNDARY OPERATORS
# ─────────────────────────────────────────────────────────────────────────────

def build_delta0(vertices, edges, m=1):
    nv, ne = len(vertices), len(edges)
    v_idx  = {v: i for i, v in enumerate(vertices)}
    rows, cols, data = [], [], []
    for k, (i, j) in enumerate(edges):
        vi, vj = v_idx[i], v_idx[j]
        for d in range(m):
            rows += [k*m+d, k*m+d]
            cols += [vj*m+d, vi*m+d]
            data += [+1.0,   -1.0]
    return sp.csr_matrix((data, (rows, cols)), shape=(ne*m, nv*m))


def build_delta1(edges, triangles, m=1):
    ne, nt = len(edges), len(triangles)
    e_idx  = {e: i for i, e in enumerate(edges)}
    rows, cols, data = [], [], []
    for t, (i, j, k) in enumerate(triangles):
        for (a, b), sgn in [((i,j),+1), ((j,k),+1), ((i,k),-1)]:
            eid = e_idx[tuple(sorted((a, b)))]
            for d in range(m):
                rows.append(t*m+d); cols.append(eid*m+d)
                data.append(float(sgn))
    return sp.csr_matrix((data, (rows, cols)), shape=(nt*m, ne*m))


# ─────────────────────────────────────────────────────────────────────────────
# 3.  HODGE LAPLACIAN + HARMONIC PROJECTION
# ─────────────────────────────────────────────────────────────────────────────

def hodge_laplacian(delta0, delta1, sigma2):
    """
    Δ1 = σ⁻⁴ δ0 δ0ᵀ + δ1ᵀ δ1   (acts on the edge space, dense output).
    """
    s4     = sigma2**(-2)
    L_down = s4 * (delta0 @ delta0.T)
    L_up   = delta1.T @ delta1
    return (L_down + L_up).toarray()


def harmonic_projection(y, delta0, delta1, sigma2, tol=1e-6):
    """
    Π_{ker Δ1}(y) via eigendecomposition + Gram–Schmidt.
    Returns h_hat, S_orth (ONB of ker Δ1), d = dim ker Δ1.
    """
    D1              = hodge_laplacian(delta0, delta1, sigma2)
    eigvals, eigvecs = la.eigh(D1)
    scale            = max(np.abs(eigvals).max(), 1.0)
    ker_idx          = np.abs(eigvals) < tol * scale
    S = eigvecs[:, ker_idx]
    d = S.shape[1]

    ne_m = delta0.shape[0]
    if d == 0:
        return np.zeros(ne_m), np.zeros((ne_m, 0)), 0

    # Standard Gram–Schmidt (inner product is standard; ONB is the same)
    S_orth = np.zeros_like(S)
    for k in range(d):
        vec = S[:, k].copy()
        for j in range(k):
            vec -= np.dot(vec, S_orth[:, j]) * S_orth[:, j]
        nrm = np.linalg.norm(vec)
        S_orth[:, k] = vec / nrm if nrm > 1e-14 else vec

    h_hat = S_orth @ (S_orth.T @ y)
    return h_hat, S_orth, d


# ─────────────────────────────────────────────────────────────────────────────
# 4.  SYNTHETIC COMMUNITY GENERATOR  (Algorithm 1)
# ─────────────────────────────────────────────────────────────────────────────

def synthetic_community_generator(n, m, p_edge, n_triangles,
                                  lambda_target, tau, sigma_noise, seed):
    """
    Algorithm 1.  lambda_target = ||h*||²  (raw norm squared, NOT /σ²).
    Returns K, δ0, δ1, σ², φ*, h*, x*, y.
    """
    rng = np.random.default_rng(seed)

    vertices, edges, triangles, b1 = build_complex(n, p_edge, n_triangles, rng)
    if b1 < 1:
        raise ValueError(f"b1={b1}: harmonic space is trivial.")
    nv, ne = len(vertices), len(edges)

    phi_star = rng.standard_normal((nv, m)) * tau
    delta0   = build_delta0(vertices, edges, m)
    delta1   = build_delta1(edges, triangles, m)
    sigma2   = sigma_noise**2

    raw = rng.standard_normal(ne * m)
    h_raw, S_orth, d = harmonic_projection(raw, delta0, delta1, sigma2)
    if d == 0:
        raise ValueError("Trivial harmonic space.")

    norm_sq = np.dot(h_raw, h_raw)
    h_star  = h_raw * np.sqrt(lambda_target / norm_sq)

    x_star = delta0 @ phi_star.flatten() + h_star
    y      = x_star + rng.standard_normal(ne * m) * sigma_noise

    K = dict(vertices=vertices, edges=edges, triangles=triangles, b1=b1)
    return K, delta0, delta1, sigma2, phi_star, h_star, x_star, y


# ─────────────────────────────────────────────────────────────────────────────
# 5.  COUPLING RESIDUAL ESTIMATOR
#
#  R_hat = ||h_hat||²         (recovered norm²)
#  R_target = ||h*||²         (injected norm²)
#  E[R_hat] = R_target + b1·σ²   (non-central chi² mean, standard result)
# ─────────────────────────────────────────────────────────────────────────────

def one_replicate(n, p_edge, n_triangles, R_target, sigma_noise, seed,
                  tau=1.0, m=1):
    """
    One Monte Carlo replicate. Returns R_hat = ||h_hat||².
    """
    K, delta0, delta1, sigma2, phi, h_star, x_star, y = \
        synthetic_community_generator(
            n=n, m=m, p_edge=p_edge, n_triangles=n_triangles,
            lambda_target=R_target, tau=tau,
            sigma_noise=sigma_noise, seed=seed)

    h_hat, _, _ = harmonic_projection(y, delta0, delta1, sigma2)
    return np.dot(h_hat, h_hat), K['b1']


# ─────────────────────────────────────────────────────────────────────────────
# 6.  MONTE CARLO EXPERIMENT
# ─────────────────────────────────────────────────────────────────────────────

def run_experiment(n=10, p_edge=0.45, n_triangles=3,
                   sigma_noise=0.3, n_rep=60, base_seed=42):
    """
    For each R_target ∈ R_grid: run n_rep replicates, collect mean ± 95% CI.
    Returns R_grid, means, ci_halves, d_r_sigma2_estimate.
    """
    R_grid  = [0.0, 0.1, 0.2, 0.4, 0.6, 0.8, 1.0]
    t_crit  = stats.t.ppf(0.975, df=n_rep - 1)
    means, ci_halves, b1_vals = [], [], []

    for R_target in R_grid:
        samples, b1s = [], []
        for rep in range(n_rep):
            seed = base_seed + rep * 137 + int(R_target * 1000)
            try:
                rhat, b1 = one_replicate(
                    n=n, p_edge=p_edge, n_triangles=n_triangles,
                    R_target=R_target, sigma_noise=sigma_noise, seed=seed)
                samples.append(rhat)
                b1s.append(b1)
            except ValueError:
                pass

        arr = np.array(samples)
        if len(arr) < 2:
            means.append(np.nan); ci_halves.append(np.nan)
            print(f"  R_target={R_target:.2f}  SKIPPED (n_valid={len(arr)})")
        else:
            m_  = arr.mean()
            ci  = t_crit * arr.std(ddof=1) / np.sqrt(len(arr))
            means.append(m_); ci_halves.append(ci)
            b1_vals.extend(b1s)
            print(f"  R_target={R_target:.2f}  R_hat={m_:.4f} ± {ci:.4f}"
                  f"  (n_valid={len(arr)}, b1≈{int(np.mean(b1s))})")

    # Theoretical offset = b1 * sigma²
    d_r_sigma2 = np.mean(b1_vals) * sigma_noise**2
    return R_grid, np.array(means), np.array(ci_halves), d_r_sigma2


# ─────────────────────────────────────────────────────────────────────────────
# 7.  FIGURE
# ─────────────────────────────────────────────────────────────────────────────

def make_figure(R_grid, means, ci_halves, d_r_sigma2, out_path):
    R_target = np.array(R_grid)
    R_fine   = np.linspace(-0.04, max(R_grid) + 0.08, 300)

    # Estimated offset from data (should equal d_r·σ²)
    valid    = ~np.isnan(means)
    offset   = float(np.mean(means[valid] - R_target[valid]))
    exp_line = R_fine + offset

    fig, ax = plt.subplots(figsize=(6.5, 5.2))
    fig.patch.set_facecolor('#F7F7F9')
    ax.set_facecolor('#F7F7F9')
    ax.grid(True, color='white', linewidth=1.1, zorder=0)
    ax.set_axisbelow(True)

    # Identity (exact recovery)
    ax.plot(R_fine, R_fine, color='#888888', linewidth=1.5,
            linestyle='--', zorder=2, label='Identity (exact recovery)')

    # Theoretical  E[R_hat] = R + d_r·σ²
    ax.plot(R_fine, exp_line, color='#C0392B', linewidth=1.8,
            linestyle='-.', zorder=3,
            label=r'Theoretical $\mathbb{E}[\hat{R}]=R+d_r\sigma^2$')

    # Confidence band
    mask = valid
    ax.fill_between(R_target[mask],
                    means[mask] - ci_halves[mask],
                    means[mask] + ci_halves[mask],
                    color='#7B5EA7', alpha=0.14, zorder=1)

    # Data points
    ax.errorbar(R_target[mask], means[mask],
                yerr=ci_halves[mask], fmt='o',
                color='#4A3580', markerfacecolor='#4A3580',
                markeredgecolor='white', markeredgewidth=0.8,
                markersize=7, ecolor='#4A3580', elinewidth=1.5,
                capsize=4, capthick=1.5, zorder=5,
                label=r'Recovered $\hat{R}$ (mean $\pm$ 95% CI, $n=60$)')

    # Offset annotation
    mid_x = 0.45
    ax.annotate(
        r'Offset $\approx d_r\sigma^2 \approx {:.3f}$'.format(offset),
        xy=(mid_x, mid_x + offset),
        xytext=(mid_x + 0.06, max(mid_x + offset - 0.25, 0.05)),
        fontsize=8.5, color='#C0392B',
        arrowprops=dict(arrowstyle='->', color='#C0392B', lw=1.2),
        ha='left')

    ax.set_xlabel(r'Injected coupling residual $R_{\mathrm{target}}$',
                  fontsize=11, labelpad=6)
    ax.set_ylabel(r'Recovered coupling residual $\hat{R}$',
                  fontsize=11, labelpad=6)
    ax.set_xlim(-0.06, max(R_grid) + 0.10)
    y_max = float(np.nanmax(means + ci_halves)) * 1.12
    y_min = min(float(np.nanmin(means - ci_halves)) * 1.05, -0.05)
    ax.set_ylim(y_min, y_max)
    ax.set_xticks(R_grid)
    ax.tick_params(axis='both', labelsize=9.5, length=3)

    for s in ['top', 'right']:
        ax.spines[s].set_visible(False)
    for s in ['left', 'bottom']:
        ax.spines[s].set_color('#CCCCCC')
        ax.spines[s].set_linewidth(0.8)

    leg = ax.legend(fontsize=8.5, loc='upper left', frameon=True,
                    framealpha=0.88, edgecolor='#CCCCCC',
                    handlelength=2.2, borderpad=0.7, labelspacing=0.55)
    leg.get_frame().set_facecolor('#F7F7F9')
    #ax.set_title(
    #    'Validation of the Künneth coupling decomposition (Theorem 2)',
    #    fontsize=9.5, pad=8, color='#333333', fontweight='normal')

    plt.tight_layout(pad=1.2)
    plt.savefig(out_path, dpi=300, bbox_inches='tight',
                facecolor=fig.get_facecolor())
    print(f"\nFigure saved → {out_path}")


# ─────────────────────────────────────────────────────────────────────────────
# 8.  MAIN
# ─────────────────────────────────────────────────────────────────────────────

if __name__ == '__main__':
    print("Running Monte Carlo experiment (n_rep=60 per R_target)…\n")
    R_grid, means, ci_halves, d_r_sigma2 = run_experiment(
        n           = 10,     # species (vertices)
        p_edge      = 0.45,   # ER edge probability
        n_triangles = 3,      # triangles to include (keeps b1 manageable)
        sigma_noise = 0.3,    # observation noise σ
        n_rep       = 60,     # Monte Carlo replicates per R_target
        base_seed   = 42
    )
    print(f"\nTheoretical offset d_r·σ² ≈ {d_r_sigma2:.4f}")

    make_figure(R_grid, means, ci_halves, d_r_sigma2,
                out_path='fig3_coupling_decomposition.png')